# Evaluation of Experiments

This notebook contains the evaluation of experiments concerning the publication

__BoSS: An Ensemble-Driven Oracle Strategy for Deep Active Image Classification__

It contains diverse plots and tables for the following experiments:

- Baselines + Oracle Model (DINOV2)
- Baselines + Oracle Model (SWINV2)

In [ ]:
!rsync -avz cluster.ies:/mnt/stud/work/phahn/repositories/dal-toolbox/perfdal.db /home/phahn/repositories/dal-toolbox/publications/perf_dal/notebooks/paul/perfdal.db

In [ ]:
# Some imports and general information

import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import MaxNLocator

############### TODO #################
# Setting paths
uri = 'sqlite:///perfdal.db'
save_dir = '/home/phahn/repositories/dal-toolbox/publications/perf_dal/plots/'
######################################

# Define a custom style dictionary
custom_style = {
    # Figure and axes
    "figure.figsize": (10, 6),
    "figure.dpi": 100,
    "axes.facecolor": "white",
    "axes.edgecolor": "lightgray",

    # Lines and markers
    "lines.linewidth": 1.5,
    "lines.markersize": 6,

    # Font settings
    #"font.family": "sans-serif",
    #"font.sans-serif": ["DejaVu Sans"],

    # Save figure settings
    "savefig.dpi": 300,
}

# Update matplotlib's global rcParams with your custom style
mpl.rcParams.update(custom_style)
mpl.rcParams['axes.grid'] = True
mpl.rcParams["grid.linestyle"] = '--'
mpl.rcParams["grid.linewidth"] = .5
mpl.rcParams["grid.alpha"] = 0.7
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=["#6C8EBF", "#D83F3F", "#5C985C", "#DC7A17", "#9C7AA1", "#F8C96B", "#A2D5C6", "#E699B3"])
mpl.rcParams.update({'font.family': 'serif', 'font.size': 9, 'font.weight':'normal'})

# Datasets
datasets = {
    'cifar10': {'qs':10, 'n':'CIFAR-10'}, 
    'stl10':{'qs':10, 'n':'STL-10'}, 
    'snacks':{'qs':20, 'n':'SNACKS'}, 
    'flowers102':{'qs':25, 'n':'Flowers102'},
    'dopanim':{'qs':50, 'n':'Dopanim'},
    'dtd':{'qs':50, 'n':'DTD'}, 
    'food101':{'qs':100, 'n':'Food101'}, 
    'cifar100':{'qs':100, 'n':'CIFAR-100'},
    'tiny_imagenet':{'qs':200, 'n': 'TinyImageNet'},
    'imagenet':{'qs':1000, 'n':'ImageNet'},
    }

# Query Strategies
query_strategies = {
    'alfamix':{'n':'AlfaMix', 'c':"#6C8EBF", 'ls':'-'},
    'badge':{'n':'BADGE', 'c':"#D45D5D", 'ls':'-'},
    'bait':{'n':'BAIT', 'c':"#5C985C", 'ls':'-'},
    'coreset':{'n':'CoreSet', 'c':"#D9822B", 'ls':'-'},
    'dropquery':{'n':'DropQuery', 'c':"#9C7AA1", 'ls':'-'},
    'margin':{'n':'Margin', 'c':"#DAB66C", 'ls':'-'},
    'random':{'n':'Random', 'c':"black", 'ls':'-'},
    'typiclust':{'n':'Typiclust', 'c':"#E699B3", 'ls':'-'},

    'Oracle':{'n':'BoSS', 'c':'black', 'ls':'-.'},

    'Oracle (S)':{'n':'Oracle (S)', 'c':'black', 'ls':':'},
    'Oracle (M)':{'n':'Oracle (M)', 'c':'black', 'ls':'-.'},
    'Oracle (L)':{'n':'Oracle (L)', 'c':'black', 'ls':'--'},
    'Oracle (XL)':{'n':'Oracle (XL)', 'c':'black', 'ls':'-.'},

    'Oracle (zero_one)':{'n':'Oracle (Zero One)', 'c':'black', 'ls':'-.'},
    'Oracle (brier)':{'n':'Oracle (Brier)', 'c':'green', 'ls':'-'},
    'Oracle (cross_entropy)':{'n':'Oracle (Cross Entropy)', 'c':'orange', 'ls':'-'},

    'Oracle (RTE=5)':{'n':'Oracle (5)', 'c':"#6C8EBF", 'ls':'-'},
    'Oracle (RTE=10)':{'n':'Oracle (10)', 'c':"#D45D5D", 'ls':'-'},
    'Oracle (RTE=25)':{'n':'Oracle (25)', 'c':"#5C985C", 'ls':'-'},
    'Oracle (RTE=50)':{'n':'Oracle (50)', 'c':'black', 'ls':'-.'},
    'Oracle (RTE=100)':{'n':'Oracle (100)', 'c':"#D9822B", 'ls':'-'},
    'Oracle (RTE=200)':{'n':'Oracle (200)', 'c':"#E699B3", 'ls':'-'},

    'Naive Oracle (10)':{'n':'Naive Oracle (10)', 'c':"#6C8EBF", 'ls':'--'},
    'Naive Oracle (50)':{'n':'Naive Oracle (50)', 'c':"#6C8EBF", 'ls':':'},
    'Naive Oracle (100)':{'n':'Naive Oracle (100)', 'c':"#6C8EBF", 'ls':'-'},
    'Naive Oracle (200)':{'n':'Naive Oracle (200)', 'c':"#6C8EBF", 'ls':'-.'},

    'Oracle (10)':{'n':'Oracle (10)', 'c':"#D9822B", 'ls':'--'},
    'Oracle (50)':{'n':'Oracle (50)', 'c':"#D9822B", 'ls':':'},
    'Oracle (100)':{'n':'Oracle (100)', 'c':'black', 'ls':'-.'},
    'Oracle (200)':{'n':'Oracle (200)', 'c':"#D9822B", 'ls':'-.'},

    'Oracle (Rep vs. Unc)' : {'n':'Oracle (Rep vs. Unc)', 'c':"#D9822B", 'ls':'-.'},
    'Oracle (OBPS)' : {'n':'Oracle (OBPS)', 'c':"#D9822B", 'ls':'-.'},
    'Oracle (No Vary)' : {'n':'Oracle (No Vary)', 'c':"#D9822B", 'ls':'-.'},
}

sampling_strategies = ['randomsampling', 'typiclust', 'dropquery', 'baitsampling', 'alfamix', 'badge', 'coreset', 'marginsampling', 'typiclass', 'dropqueryclass']
sampling_strategies_labels = ['Random', 'TypiClust', 'DropQuery', 'BAIT', 'AlfaMix', 'BADGE', 'CoreSet', 'Margin', 'TypiClust*', 'DropQuery*']

# Containers for results
all_acc_curves_strategies = {}
all_pick_choices = {}
query_times = {}

# Containers for results of swin model
swin_all_acc_curves_strategies = {}
swin_all_pick_choices = {}
swin_query_times = {}

# Mlflow args
client = mlflow.tracking.MlflowClient(tracking_uri=uri)

def style_negative(v, props=''):
    return props if v.count('-') > 1 else None

def df_style(val):
    return "font-weight: bold"

In [ ]:
# This reads in data for different experiments and removes duplicate code.

def retrieve_data(all_acc_curves_strategies, query_times, all_pick_choices, client, run, key, sampling_strategies, get_pick_choices=True):
    accs = [m.value for m in client.get_metric_history(run.info.run_id, 'accuracy')]
    qts = [m.value for m in client.get_metric_history(run.info.run_id, 'query_time')]
    dataset = run.data.params['dataset_name']
    seed = run.data.params['random_seed']
    
    if str(len(accs)-1) == run.data.params['al.num_acq']: # Sorts out runs that accidentally did not track all cycles
        # Save Accuracies
        if dataset not in all_acc_curves_strategies:
            all_acc_curves_strategies[dataset] = {}
        if key not in all_acc_curves_strategies[dataset]:
            all_acc_curves_strategies[dataset][key] = {}
        all_acc_curves_strategies[dataset][key][seed] = accs

        # Save Query Times
        if dataset not in query_times:
            query_times[dataset] = {}
        if key not in query_times[dataset]:
            query_times[dataset][key] = {}
        query_times[dataset][key][seed] = qts

        # Save Pick Choices
        if get_pick_choices:
            pick_choices = {k : client.get_metric_history(run.info.run_id, f'bought_{k}') for k in sampling_strategies}
            if dataset not in all_pick_choices:
                all_pick_choices[dataset] = {}
            if key not in all_pick_choices[dataset]:
                all_pick_choices[dataset][key] = {}
            all_pick_choices[dataset][key][seed] = pick_choices
    else:
        print('Issue with', key, dataset, 'Seed_'+run.data.params['random_seed'])
    
    return all_acc_curves_strategies, query_times, all_pick_choices

## Unnecessary long cell of functions that are used for plotting

In [ ]:
# Plotting functions for a pairwise comparison in one dset, averaged over all dsets and a global comparison
def plot_learning_curves(all_acc_curves_strategies, strats, dsets, query_strategies, datasets, baseline="random", nrows=1, ncols=2, figsize=(10,4), only_tables=False, legend_height=1.225, 
                         title_y=1., title_x=1., title_pad=14., save_dir=None, n_legend_cols=5, Labels=None):
    if not only_tables:
        fig, ax = plt.subplots(nrows=nrows, ncols=ncols, figsize=figsize, tight_layout=True)
    auc_values_rel = {}
    final_acc_rel = {}
    auc_values_abs = {}
    final_acc_abs = {}
    auc_values_abs_mean = {}
    auc_values_abs_std = {}
    for i, dset in enumerate(dsets):
        auc_values_rel[dset] = {}
        final_acc_rel[dset] = {}
        auc_values_abs[dset] = {}
        final_acc_abs[dset] = {}
        auc_values_abs_mean[dset] = {}
        auc_values_abs_std[dset] = {}

        rand_accs = list(all_acc_curves_strategies[dset][baseline].values()) 
        avg_rand_accs = np.mean(rand_accs, axis=0)
        rand_auc = np.mean(rand_accs, axis=1) * 100
        rand_final_accs = [ac[-1]*100 for ac in rand_accs]
        rand_auc_mean, rand_auc_std = np.mean(rand_auc), np.std(rand_auc) / np.sqrt(rand_auc.shape[0])
        rand_acc_mean, rand_acc_std = np.mean(rand_final_accs), np.std(rand_final_accs) / np.sqrt(len(rand_final_accs))
        auc_values_rel[dset][baseline] = str(rand_auc_mean.round(2)) + '+/-' + str(rand_auc_std.round(2))
        final_acc_rel[dset][baseline] = str(rand_acc_mean.round(2)) + '+/-' + str(rand_acc_std.round(2))
        auc_values_abs[dset][baseline] = str(rand_auc_mean.round(2)) + '+/-' + str(rand_auc_std.round(2))
        final_acc_abs[dset][baseline] = str(rand_acc_mean.round(2)) + '+/-' + str(rand_acc_std.round(2))
        auc_values_abs_mean[dset][baseline] = rand_auc_mean
        auc_values_abs_std[dset][baseline] = rand_auc_std
        if not only_tables:
            if nrows == 1:
                plt.axes(ax[i])
            else:
                plt.axes(ax[i//ncols][i%ncols])
            
        n_labeled_samples = [j*datasets[dset]['qs'] + datasets[dset]['qs'] for j in range(21)]
        for qs in all_acc_curves_strategies[dset]:
            if qs in strats:
                accs = list(all_acc_curves_strategies[dset][qs].values())
                avg_accs = np.mean(accs, axis=0)
                final_accs = [ac[-1]*100 for ac in accs]
                aucs = np.mean(accs, axis=1) * 100
                aucs_mean, aucs_std  = np.mean(aucs), np.std(aucs) / np.sqrt(aucs.shape[0])
                final_acc_mean, final_acc_std = np.mean(final_accs), np.std(final_accs) / np.sqrt(len(final_accs))

                if qs != baseline:
                    auc_values_rel[dset][qs] = str((aucs_mean - rand_auc_mean).round(2)) + '+/-' + str(aucs_std.round(2))
                    final_acc_rel[dset][qs] = str((final_acc_mean - rand_acc_mean).round(2)) + '+/-' + str(final_acc_std.round(2))
                    auc_values_abs[dset][qs] = str((aucs_mean).round(2)) + '+/-' + str(aucs_std.round(2))
                    final_acc_abs[dset][qs] = str((final_acc_mean).round(2)) + '+/-' + str(final_acc_std.round(2))
                    auc_values_abs_mean[dset][qs] = aucs_mean
                    auc_values_abs_std[dset][qs] = aucs_std
                if not only_tables:
                    plt.plot(n_labeled_samples, avg_accs - avg_rand_accs, c=query_strategies[qs]['c'], label=query_strategies[qs]['n'] if not Labels else Labels[strats.index(qs)], linestyle=query_strategies[qs]['ls'])
        if not only_tables:
            if i//ncols == nrows-1:
                plt.xlabel('Number of Labels')
            if i%ncols == 0:
                plt.ylabel('Accuracy Improvement')
            plt.grid(True)
            plt.title(datasets[dset]['n'], loc='right', y=title_y, x=title_x, pad=title_pad)
            plt.xlim(datasets[dset]['qs'], 20*datasets[dset]['qs'])
            plt.ylim(-0.025, plt.ylim()[1])
            idx = [0, 4, 9, 14, 19]
            plt.xticks(ticks=[n_labeled_samples[i] for i in idx], labels=[n_labeled_samples[i] for i in idx])
            plt.gca().yaxis.set_major_locator(MaxNLocator(nbins=5))
    if not only_tables:
        handles, labels = plt.gca().get_legend_handles_labels()
        fig.legend(loc="upper center", bbox_to_anchor=(0.5, legend_height), ncol=n_legend_cols, handles=handles, labels=labels)
        if save_dir:
            plt.savefig(save_dir, bbox_inches='tight')
        plt.show()
    return {
        'acc_rel':final_acc_rel,
        'acc_abs':final_acc_abs,
        'auc_rel':auc_values_rel,
        'auc_abs':auc_values_abs,
        'auc_abs_mean':auc_values_abs_mean,
        'auc_abs_std':auc_values_abs_std
    }



def plot_auc_table(auc_mean, auc_std, baseline=None):
    # Get union of all models and datasets
    all_datasets = sorted(set(auc_mean.keys()) | set(auc_std.keys()))
    all_models = sorted({model for dataset in (auc_mean | auc_std).values() for model in dataset})

    # Move Baseline on top if given
    if baseline:
        all_models.remove(baseline)
        all_models = [baseline] + all_models

    # Create the DataFrame
    df = pd.DataFrame(index=all_models, columns=all_datasets)

    # Fill with formatted mean ± std
    for dataset in all_datasets:
        for model in all_models:
            mean = auc_mean.get(dataset, {}).get(model, None)
            std = auc_std.get(dataset, {}).get(model, None)
            if mean is not None and std is not None:
                df.loc[model, dataset] = f"{mean:.3f} ± {std:.3f}"
            else:
                df.loc[model, dataset] = "N/A"
    if baseline:
        # Build a DataFrame of styles (same shape)
        def style_func(df):
            styles = pd.DataFrame('', index=df.index, columns=df.columns)
            for row_label in df.index:
                for col_label in df.columns:
                    if row_label != baseline:
                        current = auc_mean[col_label][row_label]
                        baseline_val = auc_mean[col_label][baseline]
                        if current < baseline_val:
                            styles.loc[row_label, col_label] = 'color: red'
            return styles
        df = df.style.apply(style_func, axis=None)

        # Also make baseline row bold
        baseline_row = pd.IndexSlice[df.index[df.index == baseline], :]
        df = df.map(df_style, subset=baseline_row)
    # Show table
    display(df)
    return None


def plot_global_pairwise_comparison(results, dsets, strategies, Labels, figsize=(6,5)):
    # Compute win rates
    win_rates = pd.DataFrame(index=strategies, columns=strategies)
    for strat1 in strategies:
        for strat2 in strategies:
            if strat1 == strat2:
                win_rates.loc[strat1, strat2] = 0.0
            else:
                accs1, accs2 = [], []
                for dset in dsets:
                    for key in results[dset][strat1]:
                        accs1.append(results[dset][strat1][key])
                        accs2.append(results[dset][strat2][key])
                means1 = np.mean(accs1, axis=1)
                means2 = np.mean(accs2, axis=1)

                assert means1.shape == means2.shape, f'On {dset}, {strat1} and {strat2} do not have the same number of results'
                
                comparisons = means1.round(4) > means2.round(4)
                wins = np.sum(comparisons) / means1.shape[0]
                win_rates.loc[strat1, strat2] = wins
    win_rates.index = [strat.capitalize() for strat in win_rates.index]
    win_rates.columns = [strat.capitalize() for strat in win_rates.columns]

    # Plot the heatmap
    #win_rates = win_rates.sort_index(axis=0)
    #win_rates = win_rates.sort_index(axis=1)
    plt.figure(figsize=figsize)
    sns.heatmap(win_rates.astype(float).T, annot=True, fmt=".2f", cmap="coolwarm", center=.50, xticklabels=Labels, yticklabels=Labels)
    plt.title(f"Global Pairwise Comparison")
    plt.ylabel("Challenged Strategy")
    plt.xlabel("Challenging Strategy")
    plt.tight_layout()
    plt.show() 



def plot_global_comparison(results, dsets, strategies, Labels, figsize=(6,5)):
    # Compute win rates
    win_rates = pd.DataFrame(index=dsets, columns=strategies)
    for dset in dsets:
        for strat1 in strategies:
            accs1 = []
            for key in results[dset][strat1]:
                accs1.append(results[dset][strat1][key])
            means1 = np.mean(accs1, axis=1)
            comparisons = []
            for strat2 in strategies:
                if strat2 != strat1:
                    accs2 = []
                    for key in results[dset][strat1]:
                        accs2.append(results[dset][strat2][key])
                    means2 = np.mean(accs2, axis=1)
                    comparison = means1.round(4) > means2.round(4)
                    comparisons.append(comparison)
            comp = np.prod(comparisons, axis=0)
            wins = np.sum(comp) / means1.shape[0]
            win_rates.loc[dset, strat1] = wins

    win_rates.index = [dset.capitalize() for dset in win_rates.index]
    win_rates.columns = [strat.capitalize() for strat in win_rates.columns]

    # Plot the heatmap
    #win_rates = win_rates.sort_index(axis=0)
    #win_rates = win_rates.sort_index(axis=1)

    plt.figure(figsize=figsize)
    plt.title(f"Highest AUC per Dataset (%)")
    sns.heatmap(win_rates.astype(float), annot=True, fmt=".2f", cmap="coolwarm", center=.50, xticklabels=Labels)
    plt.ylabel("Dataset")
    plt.xlabel("Strategy")
    plt.tight_layout()
    plt.show()



def plot_average_pick_choices(all_pick_choices, fig_strats, fig_dsets, sampling_strategies, figsize=(12, 5), cmap = 'viridis', n_queries = 20, title=None, set_xticks=True, y_labels=None, save_dir=None):
    n_plots = len(fig_dsets) * len(fig_strats)

    fig, ax = plt.subplots(nrows=1, ncols=n_plots + 1, figsize=figsize, width_ratios=[1 for _ in range(n_plots)]+ [.2])
    if title:
        fig.suptitle("Average Pick Choice")
    
    imshows = []
    vmin, vmax = 1, 0
    for a, dset in enumerate(fig_dsets):
        for b, strat in enumerate(fig_strats):     
            j = a + b * len(fig_dsets)
            
            image = []
            for sampl_strat in sampling_strategies:
                img = []
                for seed in all_pick_choices[dset][strat]:
                    im = []
                    for i in range(1, n_queries):
                        if sampl_strat in all_pick_choices[dset][strat][seed] and all_pick_choices[dset][strat][seed][sampl_strat] != []:
                            im.append(1 if (all_pick_choices[dset][strat][seed][sampl_strat][i].value > all_pick_choices[dset][strat][seed][sampl_strat][i-1].value) else 0)
                        else:
                            im.append(0)
                    img.append(im)
                image.append(img)
            image = np.array(image)
            img = np.mean(image, axis=1)
            img_avg = np.mean(img, axis=-1, keepdims=True)

            vmin, vmax = min(vmin, min(img_avg)), max(vmax, max(img_avg))

            c1 = ax[j].imshow(img_avg, cmap=cmap)
            imshows.append(c1)

            for (i, k), val in np.ndenumerate(img_avg):
                ax[j].text(k, i, f"{val:.2f}" if f"{val:.2f}" != "0.00" else "X", ha='center', va='center', color='white')
            if set_xticks:
                ax[j].set_xticks(ticks=[0], labels=[strat], rotation=25)
            else:
                ax[j].set_xticks(ticks=[], labels=[])

            if j == 0:
                ax[j].set_yticks(ticks=range(len(sampling_strategies)), labels=sampling_strategies if not y_labels else y_labels)
            else:
                ax[j].set_yticks([])
            ax[j].set_title(datasets[dset]['n'], rotation=25)

    for im in imshows:
        im.set_clim(vmin, min(.5, vmax))

    fig.colorbar(im, cax=ax[-1], label='Relative Pick Frequency')
    if save_dir:
        plt.savefig(save_dir, bbox_inches='tight')
    plt.show()



def plot_avg_pick_choices_dset(all_pick_choices, strat, dsets, sampling_strategies, sampling_strategies_labels, figsize=(14, 5), cmap='viridis', title=None, axes=None, colorbar=True, yticks=True):
    if axes:
        plt.axes(axes)
    else:
        plt.figure(figsize=figsize)
    if title:
        plt.title(title)        
    
    n_queries = 20
    images = []
    for dset in dsets:
        image = []
        for sampl_strat in sampling_strategies:
            img = []
            for seed in range(1, 11):
                seed = str(seed)
                im = []
                for i in range(1, n_queries+1):
                    if sampl_strat in all_pick_choices[dset][strat][seed] and len(all_pick_choices[dset][strat][seed][sampl_strat])>0:
                        if i == 0:
                            im.append(all_pick_choices[dset][strat][seed][sampl_strat][i].value)
                        else:
                            im.append(1 if (all_pick_choices[dset][strat][seed][sampl_strat][i].value > all_pick_choices[dset][strat][seed][sampl_strat][i-1].value) else 0)
                    else:
                        im.append(0)
                img.append(im)
            image.append(img)
        images.append(image)
    images = np.array(images)
    img = np.mean(np.mean(images, axis=0), axis=1)

    c1 = plt.imshow(img, cmap=cmap)

    for (i, k), val in np.ndenumerate(img):
        plt.text(k, i, f"{val:.2f}" if f"{val:.2f}" != "0.00" else "X", ha='center', va='center', color='white')

    plt.xticks(ticks=range(20), labels=range(1,21), rotation=0)
    if yticks:
        plt.yticks(ticks=range(len(sampling_strategies)), labels=sampling_strategies_labels)
    else:
        plt.yticks(ticks=[])

    if colorbar:
        plt.colorbar(c1)
    plt.grid(False)
    if not axes:
        plt.show()

    return c1



def plot_pick_choices(all_pick_choices, strat, dset, sampling_strategies, figsize=(14, 5), cmap='viridis', title=None):
    fig = plt.figure(figsize=figsize)
    if title:
        fig.suptitle(title)
    
    n_queries = 20
    
    image = []
    for sampl_strat in sampling_strategies:
        img = []
        for seed in all_pick_choices[dset][strat]:
            im = []
            for i in range(1, n_queries+1):
                if sampl_strat in all_pick_choices[dset][strat][seed] and len(all_pick_choices[dset][strat][seed][sampl_strat])>0:
                    if i == 0:
                        im.append(all_pick_choices[dset][strat][seed][sampl_strat][i].value)
                    else:
                        im.append(1 if (all_pick_choices[dset][strat][seed][sampl_strat][i].value > all_pick_choices[dset][strat][seed][sampl_strat][i-1].value) else 0)
                else:
                    im.append(0)
            img.append(im)
        image.append(img)
    image = np.array(image)
    img = np.mean(image, axis=1)

    c1 = plt.imshow(img, cmap=cmap)

    for (i, k), val in np.ndenumerate(img):
        plt.text(k, i, f"{val:.2f}" if f"{val:.2f}" != "0.00" else "X", ha='center', va='center', color='white')

    plt.xticks(ticks=range(20), labels=range(1,21), rotation=0)
    plt.yticks(ticks=range(len(sampling_strategies)), labels=sampling_strategies)

    plt.colorbar(c1)
    plt.show()

## Experiment 1 - Baselines

The first experiment contains the runs including well-known DAL-strategies on a variety of different Datasets. These include:

Datasets = [CIFAR10, CIFAR100, STL10, Snacks, DTD, Food101, Flowers102, TinyImageNet, ImageNet]

DAL Query Strategiges = [AlfaMix, BADGE, BAIT, CoreSet, DropQuery, Margin, Random, Typiclust]

In [ ]:
experiment_name = 'dinov2_baselines'
print('#####################################', experiment_name, '#####################################')
experiment_id = client.get_experiment_by_name(experiment_name).experiment_id
runs_strategies = client.search_runs(experiment_ids=experiment_id)
print('Found {} experiments for {} with expected {} experiments.'.format(len(runs_strategies), experiment_name, 10 * 10 * 8)) # n_dsets * n_seeds * n_query_strategies

for run in runs_strategies:
    if 'al.strategy' in run.data.params: # This sorts out weird empty runs in the db file
        key = run.data.params['al.strategy']
        all_acc_curves_strategies, query_times, all_pick_choices = retrieve_data(all_acc_curves_strategies, query_times, all_pick_choices, client, run, key, sampling_strategies, get_pick_choices=False)

experiment_name = 'dinov2_oracle'
print('#####################################', experiment_name, '#####################################')
experiment_id = client.get_experiment_by_name(experiment_name).experiment_id
runs_strategies = client.search_runs(experiment_ids=experiment_id)
print('Found {} experiments for {} with expected {} experiments.'.format(len(runs_strategies), experiment_name, 10 * 10)) # n_dsets * n_seeds 

for run in runs_strategies:
    if 'al.strategy' in run.data.params: # This sorts out weird empty runs in the db file
        key = 'Oracle'
        all_acc_curves_strategies, query_times, all_pick_choices = retrieve_data(all_acc_curves_strategies, query_times, all_pick_choices, client, run, key, sampling_strategies, get_pick_choices=True)

In [ ]:
strategies = ['random', 'margin', 'alfamix', 'badge', 'bait', 'dropquery', 'coreset', 'typiclust', 'Oracle'] # Strategies to show in the plot/table
Labels = [query_strategies[strat]['n'] for strat in strategies] # Names to use for the Strategies
dsets = datasets # Datasets to show in the plots
baseline = 'random' # Baseline to show learning curves in comparison to

res_dic = plot_learning_curves(all_acc_curves_strategies, strats=strategies, dsets=dsets, datasets=datasets, query_strategies=query_strategies, nrows=2, ncols=5, figsize=(14, 4), legend_height=1.095, 
                               title_y=0.95, title_x=0.95, title_pad=-14., save_dir=save_dir+'lc_relacc_sota_dinov2_new.pdf')
auc_mean, auc_std = res_dic['auc_abs_mean'], res_dic['auc_abs_std']
plot_auc_table(auc_mean, auc_std, baseline)
plot_average_pick_choices(all_pick_choices=all_pick_choices, fig_strats=['Oracle'], fig_dsets=dsets, sampling_strategies=sampling_strategies, figsize=(10, 5), y_labels=sampling_strategies_labels,
                           set_xticks=False, save_dir=save_dir+'avg_pick_choice_dsets_dinov2.pdf')

# Baselines + Oracle (SWIN-V2)

In [ ]:
# Note: For the SwinV2 model i use different dictionaries to save the results

experiment_name = 'swinv2_baselines'
print('#####################################', experiment_name, '#####################################')
experiment_id = client.get_experiment_by_name(experiment_name).experiment_id
runs_strategies = client.search_runs(experiment_ids=experiment_id)
print('Found {} experiments for {} with expected {} experiments.'.format(len(runs_strategies), experiment_name, 10 * 10 * 8)) # n_dsets * n_seeds * n_query_strategies

for run in runs_strategies:
    if 'al.strategy' in run.data.params: # This sorts out weird empty runs in the db file
        key = run.data.params['al.strategy']
        swin_all_acc_curves_strategies, swin_query_times, swin_all_pick_choices = retrieve_data(swin_all_acc_curves_strategies, swin_query_times, swin_all_pick_choices, client, run, key, sampling_strategies, get_pick_choices=False)

experiment_name = 'swinv2_oracle'
print('#####################################', experiment_name, '#####################################')
experiment_id = client.get_experiment_by_name(experiment_name).experiment_id
runs_strategies = client.search_runs(experiment_ids=experiment_id)
print('Found {} experiments for {} with expected {} experiments.'.format(len(runs_strategies), experiment_name, 10 * 10)) # n_dsets * n_seeds 

for run in runs_strategies:
    if 'al.strategy' in run.data.params: # This sorts out weird empty runs in the db file
        key = 'Oracle'
        swin_all_acc_curves_strategies, swin_query_times, swin_all_pick_choices = retrieve_data(swin_all_acc_curves_strategies, swin_query_times, swin_all_pick_choices, client, run, key, sampling_strategies, get_pick_choices=True)

In [ ]:
strategies = ['random', 'margin', 'alfamix', 'badge', 'bait', 'dropquery', 'coreset', 'typiclust', 'Oracle'] # Strategies to show in the plot/table
Labels = [query_strategies[strat]['n'] for strat in strategies] # Names to use for the Strategies
dsets = datasets # Datasets to show in the plots
baseline = 'random' # Baseline to show learning curves in comparison to

res_dic = plot_learning_curves(swin_all_acc_curves_strategies, strats=strategies, dsets=dsets, datasets=datasets, query_strategies=query_strategies, nrows=2, ncols=5, figsize=(14, 4), legend_height=1.095, 
                               title_y=0.95, title_x=0.95, title_pad=-14., save_dir=save_dir+'lc_relacc_sota_swinv2_new.pdf')
auc_mean, auc_std = res_dic['auc_abs_mean'], res_dic['auc_abs_std']
plot_auc_table(auc_mean, auc_std, baseline)
plot_average_pick_choices(all_pick_choices=swin_all_pick_choices, fig_strats=['Oracle'], fig_dsets=dsets, sampling_strategies=sampling_strategies, figsize=(10, 5), y_labels=sampling_strategies_labels,
                           set_xticks=False, save_dir=save_dir+'avg_pick_choice_dsets_swinv2.pdf')

# Average Pick Choices per Cycle Comparison

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(16, 6), constrained_layout=True)

im1 = plot_avg_pick_choices_dset(all_pick_choices, 'Oracle', datasets, sampling_strategies, sampling_strategies_labels, title='DINOv2-ViT-S/14', axes=ax[0], colorbar=False, yticks=True)
im2 = plot_avg_pick_choices_dset(swin_all_pick_choices, 'Oracle', datasets, sampling_strategies, sampling_strategies_labels, title='SwinV2-B', axes=ax[1], colorbar=False, yticks=False)
fig.colorbar(im1, ax=ax, orientation='horizontal', fraction=0.03, pad=.0, location='top', label='Pick Frequency') # here padding controlls the height of the cbar

plt.savefig(save_dir+'avg_pick_choice_cycle_both.pdf')

# Ablation Number of Batches

In [ ]:
experiment_name = 'abl_num_batches'
print('#####################################', experiment_name, '#####################################')
experiment_id = client.get_experiment_by_name(experiment_name).experiment_id
runs_strategies = client.search_runs(experiment_ids=experiment_id)
print('Found {} experiments for {} with expected {} experiments.'.format(len(runs_strategies), experiment_name, 2 * 10 * 6)) # n_dsets * n_seeds * n_query_strategies

for run in runs_strategies:
    if 'al.strategy' in run.data.params: # This sorts out weird empty runs in the db file
        n_bat = run.data.params['al.optimal.num_batches']
        oracle_type = 'Oracle' if run.data.params['al.optimal.strategies'].count(',') > 2 else 'Naive Oracle'
        key = f'{oracle_type} ({n_bat})'
        all_acc_curves_strategies, query_times, all_pick_choices = retrieve_data(all_acc_curves_strategies, query_times, all_pick_choices, client, run, key, sampling_strategies, get_pick_choices=False)

all_acc_curves_strategies['cifar10']['Oracle (100)'] = all_acc_curves_strategies['cifar10']['Oracle']
all_acc_curves_strategies['dtd']['Oracle (100)'] = all_acc_curves_strategies['dtd']['Oracle']

all_pick_choices['cifar10']['Oracle (100)'] = all_pick_choices['cifar10']['Oracle']
all_pick_choices['dtd']['Oracle (100)'] = all_pick_choices['dtd']['Oracle']


In [ ]:
strategies = ['Oracle', 'Naive Oracle (100)', 'random']
Labels = ['BoSS', 'Naive', 'Random']
dsets = ['cifar10', 'dtd']
baseline = 'random'

res_dic = plot_learning_curves(all_acc_curves_strategies, strats=strategies, dsets=dsets, datasets=datasets, query_strategies=query_strategies, nrows=1, ncols=2, figsize=(6, 3), legend_height=1.085, 
                               title_y=0.95, title_x=0.95, title_pad=-14., save_dir=save_dir+'abl_num_batches.pdf', Labels=Labels)
auc_mean, auc_std = res_dic['auc_abs_mean'], res_dic['auc_abs_std']
plot_auc_table(auc_mean, auc_std, baseline)
#plot_average_pick_choices(all_pick_choices=all_pick_choices, fig_strats=['Oracle'], fig_dsets=dsets, sampling_strategies=sampling_strategies, figsize=(10, 5), y_labels=sampling_strategies_labels,
#                           set_xticks=False, save_dir='/home/phahn/repositories/dal-toolbox/publications/perf_dal/plots/avg_pick_choice_dsets_swinv2.pdf')

# Ablation ACQ Size

In [ ]:
experiment_name = 'abl_acq_size_new'
print('#####################################', experiment_name, '#####################################')
experiment_id = client.get_experiment_by_name(experiment_name).experiment_id
runs_strategies = client.search_runs(experiment_ids=experiment_id)
print('Found {} experiments for {} with expected {} experiments.'.format(len(runs_strategies), experiment_name, 10 * 2 * 3)) # n_dsets * n_seeds * (n_acq_sizes-1)

for run in runs_strategies:
    if 'al.strategy' in run.data.params: # This sorts out weird empty runs in the db file
        acq_size = {'5':'S', '10':'M', '20':'L', '40':'XL', '25':'S', '50':'M', '100':'L', '200':'XL'}[run.data.params['al.acq_size']]
        key = f'Oracle ({acq_size})'
        all_acc_curves_strategies, query_times, all_pick_choices = retrieve_data(all_acc_curves_strategies, query_times, all_pick_choices, client, run, key, sampling_strategies, get_pick_choices=False)

# Manually also save normal oracle as M
all_acc_curves_strategies['cifar10']['Oracle (M)'] = all_acc_curves_strategies['cifar10']['Oracle']
all_acc_curves_strategies['dtd']['Oracle (M)'] = all_acc_curves_strategies['dtd']['Oracle']

all_pick_choices['cifar10']['Oracle (M)'] = all_pick_choices['cifar10']['Oracle']
all_pick_choices['dtd']['Oracle (M)'] = all_pick_choices['dtd']['Oracle']

In [ ]:
dsets = ['cifar10', 'dtd']
baseline = "Oracle (S)"
new_accs = {}
auc_values_abs = {}

for dset in dsets:
    new_accs[dset] = {}
    auc_values_abs[dset] = {}
    for i, acq_size in enumerate(['S', 'M', 'L', 'XL']):
        step_size = [8, 4, 2, 1][i]
        strat = f'Oracle ('+acq_size+')'
        accs_reduced = [[accs[j] for j in range(len(accs)) if j % step_size == 0] for accs in list(all_acc_curves_strategies[dset][strat].values())]
        auc_mean = np.mean(accs_reduced) * 100
        auc_std = np.std(np.mean(accs_reduced, axis=1)) / np.sqrt(np.mean(accs_reduced, axis=1).shape[0]) * 100
        auc_values_abs[dset][strat] = str((auc_mean).round(2)) + '+/-' + str(auc_std.round(2))
        new_accs[dset][strat] = accs_reduced
    rand_accs_reduced = [[accs[j] for j in range(len(accs)) if j % 4 == 0] for accs in list(all_acc_curves_strategies[dset][baseline].values())]
    new_accs[dset][baseline] = rand_accs_reduced

print('##################################### Test-AUCs (Absolute) #####################################')
df = pd.DataFrame(auc_values_abs)
random_row = pd.IndexSlice[df.index[df.index == baseline], :]
s2 = df.style.map(style_negative, props='color:red;').map(df_style, subset=random_row)
display(s2)

# Ablation Varying the Subset-Size

In [ ]:
experiment_name = 'abl_vary_sss'
print('#####################################', experiment_name, '#####################################')
experiment_id = client.get_experiment_by_name(experiment_name).experiment_id
runs_strategies = client.search_runs(experiment_ids=experiment_id)
print('Found {} experiments for {} with expected {} experiments.'.format(len(runs_strategies), experiment_name, 10 * 2)) # n_dsets * n_seeds * (n_perf_est-1)

for run in runs_strategies:
    if 'al.strategy' in run.data.params: # This sorts out weird empty runs in the db file
        key = 'Oracle (No Vary)'
        all_acc_curves_strategies, query_times, all_pick_choices = retrieve_data(all_acc_curves_strategies, query_times, all_pick_choices, client, run, key, sampling_strategies, get_pick_choices=True)

In [ ]:
strategies = ['Oracle', 'Oracle (No Vary)', 'random']
Labels = [query_strategies[strat]['n'] for strat in strategies]
dsets = ['cifar10', 'dtd']
baseline = 'random'

res_dic = plot_learning_curves(all_acc_curves_strategies, strats=strategies, dsets=dsets, datasets=datasets, query_strategies=query_strategies, nrows=1, ncols=2, figsize=(6, 3), legend_height=1.085, 
                               title_y=0.95, title_x=0.95, title_pad=-14., save_dir=save_dir+'learning_curves_vary_sss.pdf')
auc_mean, auc_std = res_dic['auc_abs_mean'], res_dic['auc_abs_std']
plot_auc_table(auc_mean, auc_std, baseline)
plot_average_pick_choices(all_pick_choices=all_pick_choices, fig_strats=strategies[:-1], fig_dsets=dsets, sampling_strategies=sampling_strategies, figsize=(4, 5), y_labels=sampling_strategies_labels,
                           set_xticks=True, save_dir=save_dir+'avg_pick_choice_dsets_perf_est.pdf')

# Ablation: Performance Estimation

In [ ]:
experiment_name = 'abl_perf_est'
print('#####################################', experiment_name, '#####################################')
experiment_id = client.get_experiment_by_name(experiment_name).experiment_id
runs_strategies = client.search_runs(experiment_ids=experiment_id)
print('Found {} experiments for {} with expected {} experiments.'.format(len(runs_strategies), experiment_name, 10 * 2 * 2)) # n_dsets * n_seeds * (n_perf_est-1)

for run in runs_strategies:
    if 'al.strategy' in run.data.params: # This sorts out weird empty runs in the db file
        perf_est = run.data.params['al.optimal.loss']
        key = f'Oracle ({perf_est})'
        all_acc_curves_strategies, query_times, all_pick_choices = retrieve_data(all_acc_curves_strategies, query_times, all_pick_choices, client, run, key, sampling_strategies, get_pick_choices=True)

# Manually also save normal oracle as M
all_acc_curves_strategies['cifar10']['Oracle (zero_one)'] = all_acc_curves_strategies['cifar10']['Oracle']
all_acc_curves_strategies['dtd']['Oracle (zero_one)'] = all_acc_curves_strategies['dtd']['Oracle']

all_pick_choices['cifar10']['Oracle (zero_one)'] = all_pick_choices['cifar10']['Oracle']
all_pick_choices['dtd']['Oracle (zero_one)'] = all_pick_choices['dtd']['Oracle']

In [ ]:
strategies = ['Oracle (zero_one)', 'Oracle (cross_entropy)', 'Oracle (brier)', 'random']
Labels = [query_strategies[strat]['n'] for strat in strategies]
dsets = ['cifar10', 'dtd']
baseline = 'random'

res_dic = plot_learning_curves(all_acc_curves_strategies, strats=strategies, dsets=dsets, datasets=datasets, query_strategies=query_strategies, nrows=1, ncols=2, figsize=(6, 3), legend_height=1.085, 
                               title_y=0.95, title_x=0.95, title_pad=-14., save_dir=save_dir+'learning_curves_perf_est.pdf')
auc_mean, auc_std = res_dic['auc_abs_mean'], res_dic['auc_abs_std']
plot_auc_table(auc_mean, auc_std, baseline)
plot_average_pick_choices(all_pick_choices=all_pick_choices, fig_strats=strategies[:-1], fig_dsets=dsets, sampling_strategies=sampling_strategies, figsize=(5, 5), y_labels=sampling_strategies_labels,
                           set_xticks=True, save_dir=save_dir+'avg_pick_choice_dsets_perf_est.pdf')

# Ablation: Number of Retraining Epochs

In [ ]:
experiment_name = 'abl_retraining_epochs'
print('#####################################', experiment_name, '#####################################')
experiment_id = client.get_experiment_by_name(experiment_name).experiment_id
runs_strategies = client.search_runs(experiment_ids=experiment_id)
print('Found {} experiments for {} with expected {} experiments.'.format(len(runs_strategies), experiment_name, 10 * 2 * 6)) # n_dsets * n_seeds * (n_perf_est-1)

for run in runs_strategies:
    if 'al.strategy' in run.data.params: # This sorts out weird empty runs in the db file
        rte = run.data.params['al.optimal.num_retraining_epochs']
        key = f'Oracle (RTE={rte})'
        all_acc_curves_strategies, query_times, all_pick_choices = retrieve_data(all_acc_curves_strategies, query_times, all_pick_choices, client, run, key, sampling_strategies, get_pick_choices=True)

# Manually also save normal oracle as M
all_acc_curves_strategies['cifar10']['Oracle (RTE=50)'] = all_acc_curves_strategies['cifar10']['Oracle']
all_acc_curves_strategies['dtd']['Oracle (RTE=50)'] = all_acc_curves_strategies['dtd']['Oracle']

all_pick_choices['cifar10']['Oracle (RTE=50)'] = all_pick_choices['cifar10']['Oracle']
all_pick_choices['dtd']['Oracle (RTE=50)'] = all_pick_choices['dtd']['Oracle']

In [ ]:
strategies = [f'Oracle (RTE={rte})' for rte in [5, 10, 25, 50, 100, 200]] + ['random']
Labels = [query_strategies[strat]['n'] for strat in strategies]
dsets = ['cifar10', 'dtd']
baseline = 'random'

res_dic = plot_learning_curves(all_acc_curves_strategies, strats=strategies, dsets=dsets, datasets=datasets, query_strategies=query_strategies, nrows=1, ncols=2, figsize=(6, 3), legend_height=1.15, 
                               title_y=0.95, title_x=0.95, title_pad=-14., save_dir=save_dir+'learning_curves_ret_epo.pdf', baseline=baseline, n_legend_cols=4)
auc_mean, auc_std = res_dic['auc_abs_mean'], res_dic['auc_abs_std']
plot_auc_table(auc_mean, auc_std, baseline)
plot_average_pick_choices(all_pick_choices=all_pick_choices, fig_strats=strategies[:-1], fig_dsets=dsets, sampling_strategies=sampling_strategies, figsize=(10, 5), y_labels=sampling_strategies_labels,
                           set_xticks=True, save_dir=save_dir+'avg_pick_choice_dsets_ret_ep.pdf')

# Ablation: Representation Sampling vs. Uncertainty Sampling

In [ ]:
experiment_name = 'abl_repr_vs_unc'
print('#####################################', experiment_name, '#####################################')
experiment_id = client.get_experiment_by_name(experiment_name).experiment_id
runs_strategies = client.search_runs(experiment_ids=experiment_id)
print('Found {} experiments for {} with expected {} experiments.'.format(len(runs_strategies), experiment_name, 10 * 2)) # n_dsets * n_seeds * (n_perf_est-1)

for run in runs_strategies:
    if 'al.strategy' in run.data.params: # This sorts out weird empty runs in the db file
        key = f'Oracle (Rep vs. Unc)'
        all_acc_curves_strategies, query_times, all_pick_choices = retrieve_data(all_acc_curves_strategies, query_times, all_pick_choices, client, run, key, sampling_strategies, get_pick_choices=True)

In [ ]:
plot_avg_pick_choices_dset(all_pick_choices, 'Oracle (Rep vs. Unc)', ['cifar10'], ['randomsampling', 'baitsampling', 'typiclust'], ['random', 'bait', 'typiclust'], title='DINOv2-ViT-S/14 on CIFAR10', colorbar=False, yticks=True)
plt.show()

plot_avg_pick_choices_dset(all_pick_choices, 'Oracle (Rep vs. Unc)', ['dtd'], ['randomsampling', 'baitsampling', 'typiclust'], ['random', 'bait', 'typiclust'], title='DINOv2-ViT-S/14 on DTD', colorbar=False, yticks=True)
plt.show()

# Ablation: One Batch per Strategy

In [ ]:
experiment_name = 'abl_one_batch_per_strat'
print('#####################################', experiment_name, '#####################################')
experiment_id = client.get_experiment_by_name(experiment_name).experiment_id
runs_strategies = client.search_runs(experiment_ids=experiment_id)
print('Found {} experiments for {} with expected {} experiments.'.format(len(runs_strategies), experiment_name, 10 * 2)) # n_dsets * n_seeds * (n_perf_est-1)

for run in runs_strategies:
    if 'al.strategy' in run.data.params: # This sorts out weird empty runs in the db file
        key = f'Oracle (OBPS)'
        all_acc_curves_strategies, query_times, all_pick_choices = retrieve_data(all_acc_curves_strategies, query_times, all_pick_choices, client, run, key, sampling_strategies, get_pick_choices=True)

In [ ]:
plot_avg_pick_choices_dset(all_pick_choices, 'Oracle (OBPS)', ['cifar10'], sampling_strategies=sampling_strategies, sampling_strategies_labels=sampling_strategies_labels, title='DINOv2-ViT-S/14 on CIFAR10', colorbar=False, yticks=True)
plt.show()

plot_avg_pick_choices_dset(all_pick_choices, 'Oracle (OBPS)', ['dtd'], sampling_strategies=sampling_strategies, sampling_strategies_labels=sampling_strategies_labels, title='DINOv2-ViT-S/14 on DTD', colorbar=False, yticks=True)
plt.show()

# Ablation: One Batch per Strategy II (Without Varying Subset-Size for a fairer Chance)

In [ ]:
experiment_name = 'abl_one_batch_per_strat_2'
print('#####################################', experiment_name, '#####################################')
experiment_id = client.get_experiment_by_name(experiment_name).experiment_id
runs_strategies = client.search_runs(experiment_ids=experiment_id)
print('Found {} experiments for {} with expected {} experiments.'.format(len(runs_strategies), experiment_name, 10 * 2)) # n_dsets * n_seeds * (n_perf_est-1)

for run in runs_strategies:
    if 'al.strategy' in run.data.params: # This sorts out weird empty runs in the db file
        key = f'Oracle (OBPS 2)'
        all_acc_curves_strategies, query_times, all_pick_choices = retrieve_data(all_acc_curves_strategies, query_times, all_pick_choices, client, run, key, sampling_strategies, get_pick_choices=True)

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(15, 8), constrained_layout=True)
plot_avg_pick_choices_dset(all_pick_choices, 'Oracle (OBPS)', ['cifar10'], sampling_strategies=sampling_strategies, sampling_strategies_labels=sampling_strategies_labels, title='DINOv2-ViT-S/14 on CIFAR10', colorbar=False, yticks=True, axes=ax[0][0])
plot_avg_pick_choices_dset(all_pick_choices, 'Oracle (OBPS)', ['dtd'], sampling_strategies=sampling_strategies, sampling_strategies_labels=sampling_strategies_labels, title='DINOv2-ViT-S/14 on DTD', colorbar=False, yticks=False, axes=ax[0][1])
plot_avg_pick_choices_dset(all_pick_choices, 'Oracle (OBPS 2)', ['cifar10'], sampling_strategies=sampling_strategies, sampling_strategies_labels=sampling_strategies_labels, title='DINOv2-ViT-S/14 on CIFAR10 (No Vary)', colorbar=False, yticks=True, axes=ax[1][0])
plot_avg_pick_choices_dset(all_pick_choices, 'Oracle (OBPS 2)', ['dtd'], sampling_strategies=sampling_strategies, sampling_strategies_labels=sampling_strategies_labels, title='DINOv2-ViT-S/14 on DTD (No Vary)', colorbar=False, yticks=False, axes=ax[1][1])
plt.show()